# Análisis del piloto UNSL — Notebook de trabajo

Este notebook acompaña el análisis empírico del capítulo de tesis. Trabaja sobre un dataset exportado con `AcademicExporter` + `RealCohortDataSource`.

**Estructura:**
1. Carga del dataset JSON y validación básica
2. Descriptivos globales y por comisión
3. Análisis longitudinal — trayectorias individuales y agregadas
4. Análisis de coherencias — matriz de correlación + PCA
5. Kappa inter-rater — importar ratings humanos y calcular agreement
6. Tests de hipótesis (McNemar para H1, regresión logística para H3)

**Datos sintéticos por defecto:** al correr este notebook sin dataset real, genera datos sintéticos representativos para validar el pipeline de análisis antes del piloto real.

## 0. Setup

In [ ]:
import json
import random
from datetime import UTC, datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# Reproducibilidad
random.seed(42)
np.random.seed(42)

# Figuras más lindas
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.family"] = "DejaVu Sans"

APPROPRIATION_ORDER = ["delegacion_pasiva", "apropiacion_superficial", "apropiacion_reflexiva"]
APPROPRIATION_ORDINAL = {a: i for i, a in enumerate(APPROPRIATION_ORDER)}
APPROPRIATION_COLORS = {
    "delegacion_pasiva": "#dc2626",
    "apropiacion_superficial": "#f59e0b",
    "apropiacion_reflexiva": "#16a34a",
}

## 1. Carga del dataset

El dataset real se genera con:

```python
from platform_ops import AcademicExporter, RealCohortDataSource
# ... ver docs/pilot/README.md
```

Por defecto acá usamos un dataset sintético. Para usar datos reales, reemplazá la ruta.

In [ ]:
REAL_DATASET_PATH = Path("./dataset_unsl_2026.json")  # cambiar si hay real


def generate_synthetic_dataset(n_students_per_comision=60, episodes_per_student_avg=8):
    """Genera un dataset sintético con estructura idéntica al exportado.

    Simula la hipótesis H1 — una mayoría progresa, una minoría no.
    """
    comisiones = {"P1": "aaaa" * 9, "P2": "bbbb" * 9, "TSU": "cccc" * 9}
    episodes = []

    for com_name, com_id in comisiones.items():
        for s in range(n_students_per_comision):
            student_alias = f"s_{com_name}_{s:03d}"
            # Perfil del estudiante: inicio + tendencia + ruido
            if s < int(n_students_per_comision * 0.6):
                # 60% mejora
                start, end = 0.3, 1.7
            elif s < int(n_students_per_comision * 0.85):
                # 25% estable
                start = end = random.uniform(0.8, 1.3)
            else:
                # 15% empeora
                start, end = 1.5, 0.4

            n_episodes = max(3, int(np.random.poisson(episodes_per_student_avg)))
            base_time = datetime(2026, 3, 1, tzinfo=UTC)

            for i in range(n_episodes):
                progress = i / max(1, n_episodes - 1)
                score = start + (end - start) * progress + np.random.normal(0, 0.3)
                score = max(0, min(2, score))
                appropriation_idx = round(score)
                appropriation = APPROPRIATION_ORDER[appropriation_idx]

                # Coherencias correlacionadas con la categoría
                ct = 0.3 + 0.25 * appropriation_idx + np.random.normal(0, 0.08)
                ccd_mean = 0.4 + 0.2 * appropriation_idx + np.random.normal(0, 0.1)
                ccd_orphan = max(0, 0.7 - 0.25 * appropriation_idx + np.random.normal(0, 0.1))
                cii_stability = 0.3 + 0.2 * appropriation_idx + np.random.normal(0, 0.1)
                cii_evolution = 0.2 + 0.25 * appropriation_idx + np.random.normal(0, 0.1)

                opened = base_time + timedelta(days=7 * i + random.randint(0, 3))
                closed = opened + timedelta(minutes=random.randint(20, 90))

                episodes.append(
                    {
                        "episode_alias": f"e_{com_name}_{s:03d}_{i}",
                        "student_alias": student_alias,
                        "comision_id": com_id,
                        "comision_name": com_name,  # helper para análisis
                        "opened_at": opened.isoformat().replace("+00:00", "Z"),
                        "closed_at": closed.isoformat().replace("+00:00", "Z"),
                        "duration_seconds": (closed - opened).total_seconds(),
                        "total_events": random.randint(5, 20),
                        "classifier_config_hash": "d" * 64,
                        "appropriation": appropriation,
                        "coherences": {
                            "ct_summary": round(max(0, min(1, ct)), 3),
                            "ccd_mean": round(max(0, min(1, ccd_mean)), 3),
                            "ccd_orphan_ratio": round(max(0, min(1, ccd_orphan)), 3),
                            "cii_stability": round(max(0, min(1, cii_stability)), 3),
                            "cii_evolution": round(max(0, min(1, cii_evolution)), 3),
                        },
                        "event_counts": {
                            "prompts": random.randint(3, 15),
                            "code_executions": random.randint(0, 10),
                            "annotations": random.randint(0, 5),
                        },
                        "prompts": [],
                    }
                )

    return {
        "cohort_alias": "UNSL_2026_P1",
        "exported_at": datetime.now(UTC).isoformat().replace("+00:00", "Z"),
        "period": {"from": "2026-03-01T00:00:00Z", "to": "2026-07-01T00:00:00Z"},
        "schema_version": "1.0.0",
        "salt_hash": "abc1234567890abc",
        "total_episodes": len(episodes),
        "total_students": n_students_per_comision * 3,
        "episodes": episodes,
    }


if REAL_DATASET_PATH.exists():
    print(f"Cargando dataset real de {REAL_DATASET_PATH}")
    with open(REAL_DATASET_PATH) as f:
        dataset = json.load(f)
else:
    print("Dataset real no encontrado, generando sintético para validar pipeline")
    dataset = generate_synthetic_dataset()

print(f"\nCohort: {dataset['cohort_alias']}")
print(f"Schema: {dataset['schema_version']}")
print(f"Total episodios: {dataset['total_episodes']}")
print(f"Total estudiantes: {dataset['total_students']}")
print(f"Salt hash (publicable): {dataset['salt_hash']}")

In [ ]:
# Flatten a DataFrame para análisis tabular
rows = []
for ep in dataset["episodes"]:
    row = {
        "episode_alias": ep["episode_alias"],
        "student_alias": ep["student_alias"],
        "comision_name": ep.get("comision_name", "UNKNOWN"),
        "opened_at": ep["opened_at"],
        "duration_seconds": ep["duration_seconds"],
        "appropriation": ep["appropriation"],
        "appropriation_ordinal": APPROPRIATION_ORDINAL[ep["appropriation"]],
        **ep["coherences"],
        **{f"count_{k}": v for k, v in ep["event_counts"].items()},
    }
    rows.append(row)

df = pd.DataFrame(rows)
df["opened_at"] = pd.to_datetime(df["opened_at"])
df = df.sort_values("opened_at").reset_index(drop=True)

print(f"DataFrame shape: {df.shape}")
df.head()

## 2. Descriptivos globales

Primer paso obligatorio: entender la forma de los datos antes de tests formales.

In [ ]:
# Distribución global de N4
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Global
global_dist = df["appropriation"].value_counts().reindex(APPROPRIATION_ORDER)
axes[0].bar(
    range(3), global_dist.values, color=[APPROPRIATION_COLORS[a] for a in APPROPRIATION_ORDER]
)
axes[0].set_xticks(range(3))
axes[0].set_xticklabels([a.replace("_", "\n") for a in APPROPRIATION_ORDER], fontsize=9)
axes[0].set_title("Distribución global de N4")
axes[0].set_ylabel("Episodios")
for i, v in enumerate(global_dist.values):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

# Por comisión
by_comision = df.groupby(["comision_name", "appropriation"]).size().unstack(fill_value=0)
by_comision = by_comision[APPROPRIATION_ORDER]
by_comision.div(by_comision.sum(axis=1), axis=0).plot(
    kind="bar",
    stacked=True,
    ax=axes[1],
    color=[APPROPRIATION_COLORS[a] for a in APPROPRIATION_ORDER],
)
axes[1].set_title("Proporción N4 por cátedra")
axes[1].set_ylabel("Fracción")
axes[1].set_xlabel("Cátedra")
axes[1].legend(title="Clasificación", bbox_to_anchor=(1.02, 1))

plt.tight_layout()
plt.show()

In [ ]:
# Estadística descriptiva de coherencias por categoría N4
coherence_cols = ["ct_summary", "ccd_mean", "ccd_orphan_ratio", "cii_stability", "cii_evolution"]

desc = df.groupby("appropriation")[coherence_cols].agg(["mean", "std"]).round(3)
desc = desc.reindex(APPROPRIATION_ORDER)
desc

**Interpretación esperada:**
- `ct_summary`, `ccd_mean`, `cii_stability`, `cii_evolution` deben crecer monótonamente con el nivel de apropiación.
- `ccd_orphan_ratio` debe decrecer monótonamente (alto orphan = delegación pasiva).

Si algún gradiente se rompe, es señal de que el árbol de decisión necesita recalibración.

## 3. Análisis longitudinal

H1: ¿los estudiantes progresan a lo largo del cuatrimestre?

In [ ]:
def classify_trajectory(scores, margin=0.25):
    """Clasifica una trayectoria comparando primer y último tercio."""
    if len(scores) < 3:
        return "insuficiente"
    size = len(scores) // 3
    first_mean = np.mean(scores[:size])
    last_mean = np.mean(scores[-size:])
    if last_mean - first_mean > margin:
        return "mejorando"
    if first_mean - last_mean > margin:
        return "empeorando"
    return "estable"


# Trayectoria por estudiante
trajectories = (
    df.sort_values("opened_at")
    .groupby("student_alias")
    .agg(
        comision=("comision_name", "first"),
        n_episodes=("appropriation_ordinal", "count"),
        first_score=("appropriation_ordinal", "first"),
        last_score=("appropriation_ordinal", "last"),
        mean_score=("appropriation_ordinal", "mean"),
        scores=("appropriation_ordinal", list),
    )
)
trajectories["progression"] = trajectories["scores"].apply(classify_trajectory)

print("Distribución de progresiones:")
print(trajectories["progression"].value_counts())
print()
print("Por cátedra:")
print(pd.crosstab(trajectories["comision"], trajectories["progression"]))

In [ ]:
# Net progression ratio por cátedra
def net_progression_ratio(group):
    labels = group["progression"].value_counts()
    with_data = (group["progression"] != "insuficiente").sum()
    if with_data == 0:
        return 0.0
    return (labels.get("mejorando", 0) - labels.get("empeorando", 0)) / with_data


npr = trajectories.groupby("comision").apply(net_progression_ratio).round(3)
print("Net progression ratio por cátedra:")
print(npr)
print()
print("Meta del protocolo: ≥ 0.3 en al menos 2 de 3 cátedras.")
print(f"¿Se cumple? {(npr >= 0.3).sum()} de 3 cátedras pasan el umbral.")

In [ ]:
# Visualización: spaghetti plot de trayectorias individuales, coloreado por progresión
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
progression_colors = {
    "mejorando": "#16a34a",
    "estable": "#64748b",
    "empeorando": "#dc2626",
    "insuficiente": "#cbd5e1",
}

for ax, com in zip(axes, ["P1", "P2", "TSU"], strict=False):
    sub = trajectories[trajectories["comision"] == com]
    for _, row in sub.iterrows():
        ax.plot(
            range(len(row["scores"])),
            row["scores"],
            color=progression_colors[row["progression"]],
            alpha=0.35,
            linewidth=0.8,
        )
    ax.set_title(f"{com}  (NPR = {npr[com]:.2f})")
    ax.set_xlabel("Episodio #")
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels([a.replace("_", "\n") for a in APPROPRIATION_ORDER], fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Categoría N4")

# Leyenda compartida
handles = [
    plt.Line2D([0], [0], color=c, linewidth=2, label=l) for l, c in progression_colors.items()
]
axes[-1].legend(handles=handles, bbox_to_anchor=(1.02, 1))

plt.tight_layout()
plt.show()

## 4. Análisis de coherencias

¿Las 5 coherencias son independientes o miden dimensiones similares? PCA + correlación.

In [ ]:
# Matriz de correlación
corr = df[coherence_cols].corr().round(3)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(coherence_cols)))
ax.set_yticks(range(len(coherence_cols)))
ax.set_xticklabels(coherence_cols, rotation=45, ha="right")
ax.set_yticklabels(coherence_cols)

for i in range(len(coherence_cols)):
    for j in range(len(coherence_cols)):
        ax.text(
            j,
            i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            color="white" if abs(corr.iloc[i, j]) > 0.5 else "black",
        )

plt.colorbar(im, ax=ax, label="Correlación Pearson")
plt.title("Matriz de correlación entre coherencias")
plt.tight_layout()
plt.show()

In [ ]:
# PCA sobre las 5 coherencias
X = df[coherence_cols].values
X_scaled = StandardScaler().fit_transform(X)
pca = PCA()
scores = pca.fit_transform(X_scaled)

print("Varianza explicada por componente:")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i + 1}: {var:.3f} ({var * 100:.1f}%)")
print(f"  Acumulado hasta PC2: {pca.explained_variance_ratio_[:2].sum() * 100:.1f}%")
print()
print("Cargas (loadings) — cómo cada coherencia contribuye a PC1 y PC2:")
loadings = pd.DataFrame(pca.components_[:2].T, index=coherence_cols, columns=["PC1", "PC2"]).round(
    3
)
print(loadings)

In [ ]:
# Scatter plot del espacio PC1 × PC2 coloreado por categoría N4
fig, ax = plt.subplots(figsize=(10, 8))

for apr in APPROPRIATION_ORDER:
    mask = df["appropriation"] == apr
    ax.scatter(
        scores[mask, 0], scores[mask, 1], c=APPROPRIATION_COLORS[apr], label=apr, alpha=0.5, s=30
    )

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
ax.set_title("Espacio PCA de las 5 coherencias, coloreado por N4")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretación:** si las 3 categorías N4 se separan claramente en el espacio PC1 × PC2, significa que las coherencias capturan la distinción relevante. Si se superponen, el árbol de decisión está agregando información adicional más allá de las coherencias.

## 5. Kappa inter-rater

Validación del clasificador contra el juicio humano. Requiere un archivo `human_ratings.csv` con columnas `episode_alias`, `human_label`. El modelo la etiqueta está en `df['appropriation']`.

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix

HUMAN_RATINGS_PATH = Path("./human_ratings.csv")

if HUMAN_RATINGS_PATH.exists():
    human = pd.read_csv(HUMAN_RATINGS_PATH)
else:
    # Sintético: en el piloto esto lo hacen los docentes en el web-teacher
    # Simulamos un rater con 80% de acuerdo con el modelo
    sample = df.sample(60, random_state=42).reset_index()
    human_labels = []
    for label in sample["appropriation"]:
        if random.random() < 0.8:
            human_labels.append(label)
        else:
            # Probabilidad de error en clases adyacentes
            other = [a for a in APPROPRIATION_ORDER if a != label]
            human_labels.append(random.choice(other))
    human = pd.DataFrame(
        {
            "episode_alias": sample["episode_alias"],
            "human_label": human_labels,
        }
    )

# Merge con predicciones del modelo
merged = human.merge(df[["episode_alias", "appropriation"]], on="episode_alias")
merged = merged.rename(columns={"appropriation": "model_label"})

kappa = cohen_kappa_score(merged["model_label"], merged["human_label"])
obs_agreement = (merged["model_label"] == merged["human_label"]).mean()

print(f"N = {len(merged)} episodios etiquetados")
print(f"Acuerdo observado: {obs_agreement:.3f} ({obs_agreement * 100:.1f}%)")
print(f"Cohen's κ: {kappa:.4f}")

if kappa >= 0.81:
    interp = "casi perfecto"
elif kappa >= 0.61:
    interp = "sustancial"
elif kappa >= 0.41:
    interp = "moderado"
elif kappa >= 0.21:
    interp = "justo"
else:
    interp = "pobre"
print(f"Interpretación (Landis & Koch): {interp}")
print()
print("Meta del protocolo: κ ≥ 0.6 → ", "✓ cumple" if kappa >= 0.6 else "✗ no cumple")

In [ ]:
# Matriz de confusión
cm = confusion_matrix(merged["model_label"], merged["human_label"], labels=APPROPRIATION_ORDER)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues", aspect="auto")
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels([a.replace("_", "\n") for a in APPROPRIATION_ORDER], fontsize=9)
ax.set_yticklabels([a.replace("_", "\n") for a in APPROPRIATION_ORDER], fontsize=9)
ax.set_xlabel("Humano (rater)")
ax.set_ylabel("Modelo (clasificador)")
ax.set_title(f"Matriz de confusión (κ = {kappa:.3f})")

max_val = cm.max()
for i in range(3):
    for j in range(3):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            color="white" if cm[i, j] > max_val / 2 else "black",
            fontsize=14,
        )

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 6. Tests de hipótesis

### H1: progresión significativa durante el cuatrimestre (McNemar)

Compara la primera y última clasificación de cada estudiante que tiene ≥3 episodios.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

# Tabla 2x2: mejoró vs no mejoró en cada métrica
enough = trajectories[trajectories["n_episodes"] >= 3].copy()
enough["improved"] = enough["last_score"] > enough["first_score"]
enough["worsened"] = enough["last_score"] < enough["first_score"]

n_improved = enough["improved"].sum()
n_worsened = enough["worsened"].sum()
n_stable = ((~enough["improved"]) & (~enough["worsened"])).sum()

print(f"Estudiantes con datos suficientes: {len(enough)}")
print(f"  Mejoraron: {n_improved}")
print(f"  Empeoraron: {n_worsened}")
print(f"  Estables: {n_stable}")
print()

# McNemar test clásico
table = [[0, n_improved], [n_worsened, 0]]
result = mcnemar(table, exact=True)
print("McNemar exact test:")
print(f"  statistic = {result.statistic:.3f}")
print(f"  p-value   = {result.pvalue:.6f}")
print(f"  → {'rechazamos' if result.pvalue < 0.05 else 'no rechazamos'} H0 (α=0.05)")
print()
print("Interpretación:")
if result.pvalue < 0.05 and n_improved > n_worsened:
    print("  ✓ Hay evidencia estadística de progresión neta (H1 apoyada).")
else:
    print("  ✗ No hay evidencia de progresión neta.")

### H3: las coherencias CII predicen rendimiento

Requiere datos de notas finales. Si no están en el dataset (export no los incluye por default), solo simulamos para validar el pipeline.

In [ ]:
# Agregar coherencias a nivel estudiante (promedios)
student_features = df.groupby("student_alias").agg(
    comision=("comision_name", "first"),
    mean_cii_stability=("cii_stability", "mean"),
    mean_cii_evolution=("cii_evolution", "mean"),
    mean_ccd=("ccd_mean", "mean"),
    mean_ct=("ct_summary", "mean"),
    n_episodes=("appropriation_ordinal", "count"),
)

# Simular notas finales correlacionadas con CII (si no hay reales)
# Esta simulación asume H3 verdadera para validar el pipeline
np.random.seed(42)
student_features["final_grade"] = (
    (
        5
        + 2 * student_features["mean_cii_stability"]
        + 2 * student_features["mean_cii_evolution"]
        + np.random.normal(0, 0.8, size=len(student_features))
    )
    .clip(1, 10)
    .round(1)
)

# Binarizar: aprobó (≥6) o no
student_features["passed"] = (student_features["final_grade"] >= 6).astype(int)

# Regresión logística
features = ["mean_cii_stability", "mean_cii_evolution", "mean_ccd", "mean_ct"]
X = student_features[features]
y = student_features["passed"]

model = LogisticRegression(max_iter=1000)
scores_cv = cross_val_score(model, X, y, cv=5, scoring="roc_auc")
model.fit(X, y)

print("Predicción de aprobación desde coherencias promedio:")
print(f"  AUC ROC (cross-val): {scores_cv.mean():.3f} ± {scores_cv.std():.3f}")
print()
print("Coeficientes (odds ratio):")
for feat, coef in zip(features, model.coef_[0], strict=False):
    or_val = np.exp(coef)
    print(f"  {feat}: β = {coef:+.3f}  OR = {or_val:.2f}")
print()
print("Interpretación: OR > 1 significa que a mayor valor de la coherencia, mayor")
print("probabilidad de aprobar. H3 espera OR > 1 en CII_stability y CII_evolution.")

## 7. Export de resultados para el paper

Última celda: generar las tablas + figuras limpias que van al paper.

In [ ]:
output_dir = Path("./paper_exports")
output_dir.mkdir(exist_ok=True)

# Tabla 1: descriptivos por cátedra
tabla1 = (
    df.groupby("comision_name")
    .agg(
        n_episodes=("episode_alias", "count"),
        n_students=("student_alias", "nunique"),
        mean_duration_min=("duration_seconds", lambda x: x.mean() / 60),
        pct_reflexiva=("appropriation", lambda x: (x == "apropiacion_reflexiva").mean()),
        pct_superficial=("appropriation", lambda x: (x == "apropiacion_superficial").mean()),
        pct_delegacion=("appropriation", lambda x: (x == "delegacion_pasiva").mean()),
    )
    .round(3)
)
tabla1.to_csv(output_dir / "tabla1_descriptivos_por_catedra.csv")

# Tabla 2: progresiones
tabla2 = pd.crosstab(trajectories["comision"], trajectories["progression"])
tabla2["NPR"] = npr
tabla2.to_csv(output_dir / "tabla2_progresiones.csv")

# Tabla 3: matriz de confusión humana-modelo
tabla3 = pd.DataFrame(cm, index=APPROPRIATION_ORDER, columns=APPROPRIATION_ORDER)
tabla3.to_csv(output_dir / "tabla3_confusion_kappa.csv")

# Resumen de resultados primarios
resumen = {
    "cohort_alias": dataset["cohort_alias"],
    "salt_hash": dataset["salt_hash"],
    "n_episodes": len(df),
    "n_students": df["student_alias"].nunique(),
    "n_students_with_enough_data": len(enough),
    "kappa": float(kappa),
    "kappa_interpretation": interp,
    "mcnemar_statistic": float(result.statistic),
    "mcnemar_pvalue": float(result.pvalue),
    "net_progression_per_comision": {k: float(v) for k, v in npr.items()},
    "meta_kappa_cumple": bool(kappa >= 0.6),
    "meta_npr_cumple_en_catedras": int((npr >= 0.3).sum()),
    "auc_prediction_passed_from_coherences": float(scores_cv.mean()),
}
with open(output_dir / "resumen_resultados_primarios.json", "w") as f:
    json.dump(resumen, f, indent=2)

print(f"Exports guardados en {output_dir}/")
print()
print("Resumen de resultados primarios:")
print(json.dumps(resumen, indent=2))

## Próximos pasos del análisis

Cuando tengamos el dataset real del piloto (post-semana 16):

1. **Reemplazar `REAL_DATASET_PATH`** con la ruta al JSON exportado por el endpoint `/cohort/export/{id}/download`.
2. **Agregar `HUMAN_RATINGS_PATH`** con los ratings reales de los 3 docentes participantes.
3. **Incorporar notas finales reales** de las cátedras (no sintéticas) — pedirlas al docente via convenio de datos.
4. **Entrevistas cualitativas** — no se analizan en este notebook; se codifican separadamente con Atlas.ti siguiendo el protocolo.
5. **Validar el A/B de profiles** — si el κ inicial es bajo, correr `POST /ab-test-profiles` con profiles candidatos antes del análisis definitivo.

Este notebook es el punto de partida del capítulo empírico de la tesis. Los scripts del piloto generan los datos; este notebook los interpreta.